In [73]:
MARGIN = 10  # pixels
FONT_SIZE = 1
FONT_THICKNESS = 1
HANDEDNESS_TEXT_COLOR = (88, 205, 54)  # vibrant green
MAX_HANDS = 2
CAMERA = 0
FPS = 30
REC_TEST = False
frame_idx = 0

DATA_HEADER = [
    "th1",
    "th2",
    "th3",
    "in1",
    "in2",
    "in3",
    "mi1",
    "mi2",
    "mi3",
    "ri1",
    "ri2",
    "ri3",
    "pi1",
    "pi2",
    "pi3",
    # "pnx",
    # "pny",
    # "pnz",
    "thumb_index_tip_dist",
    "thumb_middle_tip_dist",
    "hand_side",
]


LABEL_HEADER = [
    # finger labels
    # 1 = open, 0 = folded
    "th_open",
    "in_open",
    "mi_open",
    "ri_open",
    "pi_open",
    "ti_touch",
    "tm_touch",
]

In [74]:
import cv2
import mediapipe as mp
import numpy as np

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Utils

In [75]:
import pandas as pd
from sklearn.discriminant_analysis import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

def load_dataset(csv_paths, data_header, label_header):
    dfs = []
    for csv_file in csv_paths:
        df = pd.read_csv(csv_file)
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)
    print(df[LABEL_HEADER].describe())

    X = df[data_header].values.astype(np.float32)
    y = df[label_header].values.astype(np.float32)

    return X, y

def prepare_data(X, y):
    scaler = ColumnTransformer(
        transformers=[
            ("scale", StandardScaler(), list(range(X.shape[1]-1)))
        ],
        remainder="passthrough"
    )

    scaler.fit(X)

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True
    )


    X_train = scaler.transform(X_train)
    X_val = scaler.transform(X_val)

    return X_train, X_val, y_train, y_val, scaler

In [76]:
def create_hand_detector(model_path: str = "hand_landmarker.task", num_hands: int = 1):
    """
    MediaPipe HandLandmarker 생성.
    """
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.HandLandmarkerOptions(
        base_options=base_options,
        num_hands=num_hands,
        running_mode=mp.tasks.vision.RunningMode.VIDEO,
    )
    detector = vision.HandLandmarker.create_from_options(options)
    return detector

def get_drawing_utils():
    """
    랜드마크 시각화를 위한 연결 정보 반환.
    """
    return vision.HandLandmarksConnections.HAND_CONNECTIONS

def capture_frame(cap: cv2.VideoCapture):
    """
    웹캠에서 프레임 1장을 읽어 반환.
    실패 시 (False, None) 반환.
    """
    ret, frame = cap.read()
    frame = cv2.flip(frame, 1)
    if not ret:
        return False, None
    return True, frame

def convert_bgr_to_mp_image(bgr_frame: np.ndarray):
    """
    OpenCV BGR 이미지를 RGB 및 MediaPipe Image로 변환.
    """
    rgb_frame = cv2.cvtColor(bgr_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    return rgb_frame, mp_image

def detect_hand_landmarks(detector, bgr_frame: np.ndarray):
    """
    BGR 프레임을 받아 MediaPipe HandLandmarker로 손 랜드마크 검출.
    반환:
      - rgb_frame
      - detection_result
    """
    global frame_idx
    rgb_frame, mp_image = convert_bgr_to_mp_image(bgr_frame)
    timestamp_ms = int(frame_idx * 1000 / FPS)
    # detection_result = detector.detect(mp_image)
    detection_result = detector.detect_for_video(mp_image, timestamp_ms)
    frame_idx += 1


    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    for idx in range(len(hand_landmarks_list)):
        hand_label = handedness_list[idx][0].category_name

        if hand_label == "Right":
            handedness_list[idx][0].category_name = "Left"
        elif hand_label == "Left":
            handedness_list[idx][0].category_name = "Right"

    return rgb_frame, detection_result

def draw_landmarks_on_image(rgb_image, detection_result, hand_connections):
    hand_landmarks_list = detection_result.hand_landmarks
    handedness_list = detection_result.handedness
    annotated_image = np.copy(rgb_image)
    image_h, image_w, _ = annotated_image.shape

    for idx, hand_landmarks in enumerate(hand_landmarks_list):
        # 관절 연결선 그리기
        for connection in hand_connections:
            start_idx, end_idx = connection.start, connection.end
            start_lm = hand_landmarks[start_idx]
            end_lm = hand_landmarks[end_idx]
            start_point = (int(start_lm.x * image_w), int(start_lm.y * image_h))
            end_point = (int(end_lm.x * image_w), int(end_lm.y * image_h))
            cv2.line(annotated_image, start_point, end_point, (0, 255, 255), 2)

        # 관절 포인트 그리기
        for lm in hand_landmarks:
            px = int(lm.x * image_w)
            py = int(lm.y * image_h)
            cv2.circle(annotated_image, (px, py), 4, (255, 80, 80), -1)

        if idx < len(handedness_list):
            handedness = handedness_list[idx]
            x_coordinates = [landmark.x for landmark in hand_landmarks]
            y_coordinates = [landmark.y for landmark in hand_landmarks]
            text_x = int(min(x_coordinates) * image_w)
            text_y = int(min(y_coordinates) * image_h) - MARGIN

            cv2.putText(
                annotated_image,
                f"{handedness[0].category_name}",
                (text_x, text_y),
                cv2.FONT_HERSHEY_DUPLEX,
                FONT_SIZE,
                HANDEDNESS_TEXT_COLOR,
                FONT_THICKNESS,
                cv2.LINE_AA,
            )
    return annotated_image

# MODEL

In [77]:
import tensorflow as tf
import pickle


def load_model(model_file="gesture_model.keras", scalar_file="gesture_scaler.pkl"):
    model = tf.keras.models.load_model(model_file, safe_mode=False, compile=True)
    with open(scalar_file, "rb") as f:
        scaler = pickle.load(f)

    return model, scaler



In [78]:
load_model()

(<Functional name=gesture_branch_model, built=True>,
 ColumnTransformer(remainder='passthrough',
                   transformers=[('scale', StandardScaler(),
                                  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                   14, 15, 16])]))

In [79]:
def predict_gesture(model, scaler, feature):

    x = np.array(feature, dtype=np.float32).reshape(1, -1)

    x = scaler.transform(x)

    pred = model(x, traning = False).numpy()[0]

    return pred


def predict_hand_gesture_by_dnn(detection_result, model, scalar):

    left_result = None
    right_result = None

    hand_landmarks_list = getattr(detection_result, "hand_landmarks", [])
    handedness_list = getattr(detection_result, "handedness", [])

    if not hand_landmarks_list:
        return {"left": None, "right": None}
    

    for idx in range(len(hand_landmarks_list)):
        hand_landmarks = hand_landmarks_list[idx]
        hand_label = handedness_list[idx][0].category_name

        hand_side = -1
        if hand_label == "Right":
            hand_side = 1
        elif hand_label == "Left":
            hand_side = 0
        # hand_side = hand_side * 100

        joint = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks])

        v1 = joint[[0,1,2,3,0,5,6,7,0,9,10,11,0,13,14,15,0,17,18,19],:]
        v2 = joint[[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20],:]

        v = v2 - v1
        v = v / np.linalg.norm(v, axis=1)[:, np.newaxis]

        angle = np.arccos(np.einsum(
            'nt,nt->n',
            v[[0,1,2,4,5,6,8,9,10,12,13,14,16,17,18],:],
            v[[1,2,3,5,6,7,9,10,11,13,14,15,17,18,19],:]
        ))

        angle = np.degrees(angle)

        # TODO It will be used later... maybe?..
        # v_p1 = joint[[0, 0], :]
        # v_p2 = joint[[5, 17], :]
        # v_palm = v_p2 - v_p1

        # palm_normal = np.cross(v_palm[0], v_palm[1])
        # palm_normal = palm_normal / np.linalg.norm(palm_normal)
        # palm_normal = palm_normal * 500
        # full_data = np.append(angle, palm_normal)
        full_data = angle

        palm_size = np.linalg.norm(joint[17] - joint[5]) + 1e-6

        tdi = np.linalg.norm(joint[4] - joint[8]) / palm_size
        tdm = np.linalg.norm(joint[4] - joint[12]) / palm_size

        full_data = np.append(full_data, tdi)
        full_data = np.append(full_data, tdm)
        full_data = np.append(full_data, hand_side)

        data = np.array([full_data], dtype=np.float32)
    
        # print(data)
        # TODO Change model
        # ret, results, neighbours, dist = knn.findNearest(data, 3)
        # print(results)
        # if dist[0][0] < 2000.0:
        pred = predict_gesture(model, scalar, data)
        # label = int(results[0][0])

        if hand_label == "Right":
            right_result = pred
        elif hand_label == "Left":
            left_result = pred

    return {
        "left": left_result,
        "right": right_result
    }


# Something.. be used for multi sequence key input...

In [80]:
from enum import IntEnum, Enum, auto
from dataclasses import dataclass
from turtle import right
from typing import NamedTuple

INPUT_SIZE = len(LABEL_HEADER)


class HandSide(Enum):
    LEFT = auto()
    RIGHT = auto()


class ActionType(Enum):
    KEY = auto()
    GROUP = auto()
    MODE = auto()
    RESET = auto()
    PATTERN = auto()


class Modifier(Enum):
    SHIFT = auto()


class GestureID(IntEnum):
    RESET = 0b0001000
    G1 = 0b1000000
    G2 = 0b0100000
    G3 = 0b0010000
    G4 = 0b0000100
    G5 = 0b1100000
    G6 = 0b0110000
    G7 = 0b1000100
    G8 = 0b0100100
    G9 = 0b1110000
    G10 = 0b0111000
    G11 = 0b0011100
    G12 = 0b1100100
    G13 = 0b0111100
    G14 = 0b1111100


class Binding(NamedTuple):
    kind: ActionType
    value: str


BINDINGS = {
    HandSide.LEFT: {},
    HandSide.RIGHT: {},
    GestureID.RESET: Binding(ActionType.RESET, "root"),
    GestureID.G1: Binding(ActionType.KEY, "space"),
    GestureID.G2: Binding(ActionType.KEY, "backspace"),
    GestureID.G3: Binding(ActionType.KEY, "enter"),
    GestureID.G4: Binding(ActionType.KEY, "tab"),
    GestureID.G5: Binding(ActionType.GROUP, "alpha_1"),
    GestureID.G6: Binding(ActionType.GROUP, "alpha_2"),
    GestureID.G7: Binding(ActionType.GROUP, "alpha_3"),
    GestureID.G8: Binding(ActionType.GROUP, "alpha_4"),
    GestureID.G9: Binding(ActionType.GROUP, "alpha_5"),
    GestureID.G10: Binding(ActionType.GROUP, "alpha_5"),
    GestureID.G11: Binding(ActionType.GROUP, "digit"),
    GestureID.G12: Binding(ActionType.GROUP, "num_symbol"),
    GestureID.G13: Binding(ActionType.GROUP, "symbol"),
    GestureID.G14: Binding(ActionType.GROUP, "shift_symbol"),
}

ROOT = "root"
state = ROOT


digits = {
    GestureID.G2: 1,  # 0b0100000 : 검지
    GestureID.G6: 2,  # 0b0110000 : 검지 + 중지
    GestureID.G10: 3,  # 0b0111000 : 검지 + 중지 + 약지
    GestureID.G13: 4,  # 0b0111100 : 검지 + 중지 + 약지 + 소지
}


RAW_TO_GESTURE = {

}

class GestureHID:
    RAW_TO_GESTURE = {
        # reset
        0b0001000: GestureID.RESET,
        # 1nd
        0b1000000: GestureID.G1,
        0b0100000: GestureID.G2,
        0b0010000: GestureID.G3,
        0b0000100: GestureID.G4,
        # 2nd
        0b1100000: GestureID.G5,
        0b0110000: GestureID.G6,
        0b1000100: GestureID.G7,
        0b0100100: GestureID.G8,
        # 3nd
        0b1110000: GestureID.G9,
        0b0111000: GestureID.G10,
        0b0011100: GestureID.G11,
        0b1100100: GestureID.G12,
        # 4nd
        0b0111100: GestureID.G13,
        # 5nd
        0b1111100: GestureID.G14,
    }
    LEFT_ACTION = {
        # TODO ??...
        # GestureID.G1: Binding(ActionType.PATTERN, Modifier.SHIFT),
        # Device Select
        GestureID.G2: Binding(ActionType.MODE, "1"),
        GestureID.G6: Binding(ActionType.MODE, "2"),
        GestureID.G10: Binding(ActionType.MODE, "3"),
        # Keyboard Mode
        GestureID.G8: Binding(ActionType.MODE, "digits"),
        GestureID.G13: Binding(ActionType.MODE, "alphabets"),
    }

    DIGITS_ACTION = {
        GestureID.G2: Binding(ActionType.KEY, '1'),
        GestureID.G6: Binding(ActionType.KEY, '2'),
        GestureID.G10: Binding(ActionType.KEY, '3'),
        GestureID.G13: Binding(ActionType.KEY, '4'),
        GestureID.G14: Binding(ActionType.KEY, '5'),
        GestureID.G1: Binding(ActionType.KEY, '6'),
        GestureID.G5: Binding(ActionType.KEY, '7'),
        GestureID.G7: Binding(ActionType.KEY, '8'),
        GestureID.G12: Binding(ActionType.KEY, '9'),
        GestureID.G8: Binding(ActionType.KEY, '0'),
    }

    def __init__(self):
        self._mode = None
        self._action = None

    def bits_to_int(self, bits):
        value = 0
        for b in bits:
            value = (value << 1) | int(b)
        return value
    
    def get_mode(self, left_result) -> tuple[str | None, Modifier | None]:
        gesture_bits = self.bits_to_int(left_result)
        gesture = self.RAW_TO_GESTURE.get(gesture_bits, None)

        modifier = None
        if gesture is not None and gesture & 0b1000000:
            if gesture ^ GestureID.G1 in (GestureID.G8, GestureID.G13):
                modifier = Modifier.SHIFT
                gesture ^= 0b1000000

        mode = self.LEFT_ACTION.get(gesture, None)

        return mode, modifier
    
    def get_digit(self, right_result) -> str | None:
        gesture_bits = self.bits_to_int(right_result)
        gesture = self.RAW_TO_GESTURE.get(gesture_bits, None)

        key = self.DIGITS_ACTION.get(gesture, None)
        if key != self._action:
            self._action = key
            return key

        return None
    
    def emit(self, results):
        pred_left = results["left"]
        if pred_left is None:
            pred_left = [0 for _ in LABEL_HEADER]
        pred_right = results["right"]
        if pred_right is None:
            pred_right = [0 for _ in LABEL_HEADER]

        # TODO Test
        pred_right = [1 if p > 0.6 else 0  for p in pred_right]
        pred_left = [1 if p > 0.6 else 0  for p in pred_left]

        mode, modifier = self.get_mode(pred_left)
        if mode is not None and mode.value == 'digits':
            action = self.get_digit(pred_right)
            if action is not None:
                print(action)
        

        


# TODO Build Somthing (maybe tree?)

# TODO Temperal input ()


# def emit_action(result: list):
#     global state, digits
#     key = bits_to_int(result)
#     gesture = RAW_TO_GESTURE.get(key, None)
#     if gesture is None:
#         return

#     if gesture == GestureID.RESET:
#         if not state == "root":
#             state = "root"
#             print("reset")
#         return

#     if state == "digit":
#         num = digits.get(gesture, None)
#         if num is not None:
#             print(digits[gesture])

#     if state == ROOT:
#         type, next_state = BINDINGS[gesture]
#         if state != next_state:
#             state = next_state
#             print(next_state)
#         return


# TODO Predict what is next state using built tree (Maybe State Machine)

# Result Test

In [81]:
model, scalar = load_model()

cap = cv2.VideoCapture(CAMERA)
if not cap.isOpened():
    raise RuntimeError(f"웹캠을 열 수 없습니다. camera_index={CAMERA}")

detector = create_hand_detector(model_path="hand_landmarker.task", num_hands=MAX_HANDS)
hand_connections = get_drawing_utils()

gesture_hid = GestureHID()
# knn = load_knn()
try:
    while True:
        ret, frame = capture_frame(cap)
        if not ret:
            print("프레임을 읽지 못했습니다. 종료합니다.")
            break

        rgb_frame, detection_result = detect_hand_landmarks(detector, frame)
        result = predict_hand_gesture_by_dnn(detection_result, model, scalar)
        annotated_image = draw_landmarks_on_image(rgb_frame, detection_result, hand_connections)

        # OpenCV는 BGR 기준으로 표시하므로 RGB -> BGR로 변환
        display_image = cv2.cvtColor(annotated_image, cv2.COLOR_RGB2BGR)

        font = cv2.FONT_HERSHEY_SIMPLEX
        line_height = 26

        gesture_hid.emit(result)

        # ===== LEFT HAND =====
        lines_left = ["LEFT HAND"]

        for name, p in zip(LABEL_HEADER, [0] * 7 if result['left'] is None else result['left']):
            lines_left.append(f"{name:<10} : {p:>6.2%}")

        for i, line in enumerate(lines_left):
            y = 60 + i * line_height
            cv2.putText(
                display_image,
                line,
                (10, y),
                font,
                0.6,
                (0,255,0),
                2,
                cv2.LINE_AA
            )


        # ===== RIGHT HAND =====
        lines_right = ["RIGHT HAND"]

        for name, p in zip(LABEL_HEADER, [0] * 7 if result['right'] is None else result['right']):
            lines_right.append(f"{name:<10} : {p:>6.2%}")

        for i, line in enumerate(lines_right):
            y = 60 + i * line_height
            cv2.putText(
                display_image,
                line,
                (350, y),
                font,
                0.6,
                (255,255,0),
                2,
                cv2.LINE_AA
            )


        cv2.imshow("MediaPipe Hand Landmarks", display_image)

        # ESC 또는 q 입력 시 종료
        key = cv2.waitKey(1) & 0xFF
        if key == 27 or key == ord("q"):
            break
        
finally:
    cap.release()
    cv2.waitKey(-1)
    cv2.destroyAllWindows()